# Part 4: Machine Learning with Spark ML
### Binary Classification – Term Deposit Prediction
> PySpark ML runs fully in Google Colab.

In [ ]:
!pip install pyspark -q
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import matplotlib.pyplot as plt, pandas as pd

spark = SparkSession.builder.appName('BankingML').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark ML ready ✓')

## Q1 – Data Loading and Initial Exploration

In [ ]:
from google.colab import files
uploaded = files.upload()
df = spark.read.csv('bank.csv', header=True, inferSchema=True)
print(f'Dataset: {df.count()} rows × {len(df.columns)} columns')
df.printSchema()
df.show(5)

## Q2 – Data Preprocessing

In [ ]:
# Check missing values
print('--- Missing Value Check ---')
for c in df.columns:
    n = df.filter(F.col(c).isNull()).count()
    if n > 0: print(f'  {c}: {n} nulls')
print('No null values found (dataset is clean)')

# Handle outliers in balance using IQR
q1, q3 = df.approxQuantile('balance',[0.25,0.75],0.01)
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
print(f'\nBalance IQR bounds: [{lower:.1f}, {upper:.1f}]')
df = df.withColumn('balance', F.when(F.col('balance')<lower, lower).when(F.col('balance')>upper, upper).otherwise(F.col('balance')))
print('Outliers capped (winsorization)')

# Fix pdays: -1 means never contacted → replace with 0
df = df.withColumn('pdays', F.when(F.col('pdays')==-1,0).otherwise(F.col('pdays')))

In [ ]:
# StringIndexer for categorical columns
CAT_COLS = ['job','marital','education','default','housing','loan','contact','month','poutcome']
NUM_COLS = ['age','balance','day','duration','campaign','pdays','previous']

indexers = [StringIndexer(inputCol=c,outputCol=c+'_idx',handleInvalid='keep') for c in CAT_COLS]
encoder = OneHotEncoder(inputCols=[c+'_idx' for c in CAT_COLS], outputCols=[c+'_ohe' for c in CAT_COLS])
lbl_idx = StringIndexer(inputCol='y', outputCol='label')
print('StringIndexer + OneHotEncoder configured for:', CAT_COLS)

## Q3 – Feature Engineering: VectorAssembler

In [ ]:
feature_cols = [c+'_ohe' for c in CAT_COLS] + NUM_COLS
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features_raw')
scaler = StandardScaler(inputCol='features_raw', outputCol='features', withMean=False, withStd=True)
print(f'VectorAssembler will combine {len(feature_cols)} feature groups into one vector')
print('Features:', feature_cols)

## Q4 – Model Training and Selection
**Model choice: Random Forest**
- Handles both numeric and categorical features naturally
- Provides feature importances for interpretability
- Robust against outliers and overfitting via ensemble averaging
- No assumptions about data distribution

In [ ]:
rf = RandomForestClassifier(labelCol='label', featuresCol='features', numTrees=100, maxDepth=5, seed=42)
pipeline = Pipeline(stages=indexers + [encoder, lbl_idx, assembler, scaler, rf])

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count()}  |  Test: {test_df.count()}')
print('Training Random Forest (100 trees, depth 5)...')
model = pipeline.fit(train_df)
print('Training complete ✓')

## Q5 – Model Evaluation

In [ ]:
predictions = model.transform(test_df)

auc = BinaryClassificationEvaluator(labelCol='label',rawPredictionCol='rawPrediction').evaluate(predictions)
acc = MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='accuracy').evaluate(predictions)
f1  = MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='f1').evaluate(predictions)
prec= MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='weightedPrecision').evaluate(predictions)
rec = MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='weightedRecall').evaluate(predictions)

print('='*40)
print(f'  AUC-ROC   : {auc:.4f}')
print(f'  Accuracy  : {acc:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print('='*40)

print('\n--- Confusion Matrix ---')
predictions.groupBy('label','prediction').count().orderBy('label','prediction').show()

## Q6 – Hyperparameter Tuning

In [ ]:
rf_tuned = RandomForestClassifier(labelCol='label', featuresCol='features', seed=42)
pipeline_t = Pipeline(stages=indexers+[encoder, lbl_idx, assembler, scaler, rf_tuned])

param_grid = (ParamGridBuilder()
    .addGrid(rf_tuned.numTrees, [50, 100])
    .addGrid(rf_tuned.maxDepth, [4, 6])
    .build())

evaluator = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction')
cv = CrossValidator(estimator=pipeline_t, estimatorParamMaps=param_grid,
                   evaluator=evaluator, numFolds=3, seed=42)

print('Running 3-fold CV across 4 parameter combinations...')
cv_model = cv.fit(train_df)
best_auc = max(cv_model.avgMetrics)
print(f'Best CV AUC: {best_auc:.4f}')
print(f'Best model test AUC: {evaluator.evaluate(cv_model.transform(test_df)):.4f}')

## Q7 – Feature Importances

In [ ]:
best_rf = cv_model.bestModel.stages[-1]
importances = best_rf.featureImportances.toArray()
feature_names = [c+'_ohe' for c in CAT_COLS] + NUM_COLS

fi_df = pd.DataFrame({'feature': feature_names[:len(importances)], 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(12)

print('Top 12 Feature Importances:')
print(fi_df.to_string(index=False))

plt.figure(figsize=(10,5))
plt.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color='teal')
plt.title('Top 12 Feature Importances – Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('spark_ml_feature_importances.png',dpi=150)
plt.show()

In [ ]:
spark.stop()
print('All 7 Spark ML questions complete ✓')